# 🧠 Brain Tumor Detection Using Convolutional Neural Networks

**Course**: Neural Networks and Deep Learning  
**Project**: Binary classification of brain MRI scans

---

## Overview

This notebook provides a full end-to-end demonstration of the project:

1. 📂 Dataset exploration and visualisation
2. 🏗️ Model architecture walkthrough
3. 🔄 Training with live progress
4. 📊 Evaluation: accuracy, confusion matrix, F1-score
5. 🔍 Single image prediction

---

### Course Knowledge Points Covered
| Chapter | Topic | Used In |
|---------|-------|---------|
| Ch. 2 | Loss functions, SGD, Overfitting | Training loop |
| Ch. 3 | Softmax, Cross-Entropy | Output layer + loss |
| Ch. 4 | ReLU, Feedforward networks | CNN blocks |
| Ch. 5 | Convolution, Pooling, Feature maps | BrainTumorCNN |
| Ch. 7 | Adam, BatchNorm, Dropout, Augmentation, Early Stopping | Full pipeline |


## 0. Setup & Imports

In [ ]:
import os
import sys
import random
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# Add the src directory to Python path so we can import our modules
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from model   import BrainTumorCNN
from dataset import get_dataloaders, denormalize, BrainTumorDataset

# ---- Device setup ----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'✅ Using device: {device}')
print(f'✅ PyTorch version: {torch.__version__}')

# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Configure plots
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. 📂 Dataset Exploration

**Dataset**: Brain MRI Images for Brain Tumor Detection (Kaggle)  
**Source**: https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection  
**Classes**: `no` (no tumor) and `yes` (tumor present)

Update `DATA_DIR` to point to your downloaded dataset.

In [ ]:
# ---- CONFIGURE THIS PATH ----
DATA_DIR = str(PROJECT_ROOT / 'data' / 'brain_mri')  # Contains /yes and /no folders
RESULTS_DIR = str(PROJECT_ROOT / 'results')
MODEL_PATH = os.path.join(RESULTS_DIR, 'best_model.pth')

print(f'Data directory : {DATA_DIR}')
print(f'Results dir    : {RESULTS_DIR}')

# Check the dataset exists
if os.path.exists(DATA_DIR):
    for cls in ['yes', 'no']:
        cls_path = os.path.join(DATA_DIR, cls)
        if os.path.exists(cls_path):
            count = len(list(Path(cls_path).glob('*.*')))
            print(f'  Class "{cls}": {count} images')
else:
    print('⚠️  Dataset not found. See data/README_DATASET.txt for download instructions.')

In [ ]:
# ---- Visualise sample images from each class ----

def show_sample_images(data_dir, n_per_class=6):
    """Display random samples from each class side-by-side."""
    fig, axes = plt.subplots(2, n_per_class, figsize=(n_per_class * 2.5, 5))
    fig.suptitle('Sample MRI Images — Brain Tumor Dataset', fontsize=13, fontweight='bold')
    
    for row_idx, cls in enumerate(['no', 'yes']):
        cls_dir = Path(data_dir) / cls
        if not cls_dir.exists():
            print(f'Folder not found: {cls_dir}')
            continue
        
        img_files = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        samples = random.sample(img_files, min(n_per_class, len(img_files)))
        
        label_text = 'NO TUMOR' if cls == 'no' else 'TUMOR DETECTED'
        color = 'green' if cls == 'no' else 'red'
        
        for col_idx, img_path in enumerate(samples):
            img = Image.open(img_path).convert('RGB').resize((200, 200))
            axes[row_idx][col_idx].imshow(img)
            axes[row_idx][col_idx].axis('off')
            if col_idx == 0:
                axes[row_idx][col_idx].set_title(label_text, color=color, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'sample_images.png'), bbox_inches='tight')
    plt.show()
    print('Sample images saved to results/sample_images.png')

os.makedirs(RESULTS_DIR, exist_ok=True)

if os.path.exists(DATA_DIR):
    show_sample_images(DATA_DIR)
else:
    print('Skipping visualisation — dataset not found.')

In [ ]:
# ---- Class distribution bar chart ----

if os.path.exists(DATA_DIR):
    counts = {}
    for cls in ['no', 'yes']:
        cls_dir = Path(DATA_DIR) / cls
        if cls_dir.exists():
            counts[cls] = len(list(cls_dir.glob('*.*')))
    
    fig, ax = plt.subplots(figsize=(5, 4))
    bars = ax.bar(['No Tumor', 'Tumor'], list(counts.values()),
                  color=['steelblue', 'tomato'], edgecolor='black', linewidth=0.8)
    
    for bar, count in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(count), ha='center', va='bottom', fontweight='bold')
    
    ax.set_title('Class Distribution in Dataset', fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, max(counts.values()) * 1.15)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()
    print(f'Total images: {sum(counts.values())}')

## 2. 🏗️ Model Architecture

We build a **custom CNN from scratch** — no pretrained models (ResNet, VGG, etc.).

This lets us directly connect the implementation to course theory:

```
Input (3, 224, 224)    ← RGB MRI image
  ↓
Block 1: Conv(3→32) + BN + ReLU + MaxPool   → (32, 112, 112)
Block 2: Conv(32→64) + BN + ReLU + MaxPool  → (64, 56, 56)
Block 3: Conv(64→128) + BN + ReLU + MaxPool → (128, 28, 28)
Block 4: Conv(128→256) + BN + ReLU + MaxPool→ (256, 14, 14)
  ↓
Flatten → FC(512) → Dropout(0.5) → FC(2)
  ↓
Output: [P(no_tumor), P(tumor)]
```

In [ ]:
# Build the model and print architecture
model = BrainTumorCNN(num_classes=2, dropout_rate=0.5).to(device)
model.summary()

# Verify with dummy input
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'\n✅ Forward pass OK: input {tuple(dummy.shape)} → output {tuple(out.shape)}')

In [ ]:
# ---- Visualise feature map sizes through the network ----

print('Feature map dimensions through the network:')
print('=' * 50)
x = torch.randn(1, 3, 224, 224).to(device)

model.eval()
with torch.no_grad():
    print(f'  Input          : {tuple(x.shape)}')
    x = model.block1(x);  print(f'  After Block 1  : {tuple(x.shape)}')
    x = model.block2(x);  print(f'  After Block 2  : {tuple(x.shape)}')
    x = model.block3(x);  print(f'  After Block 3  : {tuple(x.shape)}')
    x = model.block4(x);  print(f'  After Block 4  : {tuple(x.shape)}')
    flat_size = x.view(1, -1).shape[1]
    print(f'  After Flatten  : ({flat_size},)')
    x = model.classifier(x)
    print(f'  Output logits  : {tuple(x.shape)}')
print('=' * 50)

## 3. 📦 Load Data

In [ ]:
if not os.path.exists(DATA_DIR):
    print('⚠️  Dataset not found. Skipping data loading.')
    print('   See data/README_DATASET.txt for download instructions.')
else:
    train_loader, val_loader, test_loader, class_names = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=32,
        num_workers=0,
    )
    
    print(f'\nClass names   : {class_names}')
    print(f'Train batches : {len(train_loader)}')
    print(f'Val batches   : {len(val_loader)}')
    print(f'Test batches  : {len(test_loader)}')
    
    # Show one augmented batch
    images, labels = next(iter(train_loader))
    print(f'\nSample batch: images {tuple(images.shape)}, labels {labels[:8].tolist()}')

In [ ]:
# ---- Visualise augmented training batch ----

if os.path.exists(DATA_DIR):
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    fig.suptitle('Augmented Training Batch (top) vs Validation Batch (bottom)',
                 fontsize=12, fontweight='bold')
    
    train_imgs, train_lbls = next(iter(train_loader))
    val_imgs,   val_lbls   = next(iter(val_loader))
    
    for i in range(8):
        # Training (augmented)
        img_t = denormalize(train_imgs[i]).permute(1,2,0).numpy()
        axes[0][i].imshow(img_t)
        axes[0][i].set_title(class_names[train_lbls[i].item()], fontsize=8)
        axes[0][i].axis('off')
        
        # Validation (no augmentation)
        img_v = denormalize(val_imgs[i]).permute(1,2,0).numpy()
        axes[1][i].imshow(img_v)
        axes[1][i].set_title(class_names[val_lbls[i].item()], fontsize=8)
        axes[1][i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. 🔄 Training

**Configuration:**
- Optimizer: Adam (lr=0.001, weight_decay=1e-4) — Chapter 7
- Loss: CrossEntropyLoss — Chapter 3
- LR Scheduler: StepLR (step=7, gamma=0.1) — Chapter 7
- Early Stopping: patience=5 — Chapter 7
- Epochs: 25

In [ ]:
# ---- Run training ----

if not os.path.exists(DATA_DIR):
    print('⚠️  Dataset not found. Cannot train.')
else:
    # Import training utilities
    from train import train_one_epoch, validate, EarlyStopping, save_training_curves
    import torch.optim as optim
    from torch.optim.lr_scheduler import StepLR
    import time
    
    # Rebuild model fresh
    model = BrainTumorCNN(num_classes=2, dropout_rate=0.5).to(device)
    criterion     = nn.CrossEntropyLoss()
    optimizer     = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler     = StepLR(optimizer, step_size=7, gamma=0.1)
    early_stop    = EarlyStopping(patience=5)
    
    EPOCHS       = 25
    best_val_acc = 0.0
    history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print(f'Training for up to {EPOCHS} epochs on {device}...\n')
    
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc   = validate(model, val_loader, criterion, device)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        elapsed = time.time() - t0
        print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.1f}%  |  '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.1f}%  |  '
              f'{elapsed:.1f}s')
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs(RESULTS_DIR, exist_ok=True)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc, 'val_loss': val_loss,
                'class_names': class_names,
            }, MODEL_PATH)
            print(f'  → Best model saved (val_acc={val_acc*100:.2f}%)')
        
        if early_stop.step(val_loss):
            print(f'\nEarly stopping triggered at epoch {epoch}.')
            break
    
    print(f'\n✅ Training complete! Best val accuracy: {best_val_acc*100:.2f}%')

In [ ]:
# ---- Plot training curves (inline) ----

if os.path.exists(DATA_DIR) and 'history' in dir():
    epochs_ran = range(1, len(history['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Training History', fontsize=14, fontweight='bold')
    
    # Loss
    ax1.plot(epochs_ran, history['train_loss'], 'b-o', label='Train', markersize=4)
    ax1.plot(epochs_ran, history['val_loss'],   'r-o', label='Validation', markersize=4)
    ax1.set_title('Loss per Epoch')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
    ax1.legend(); ax1.grid(alpha=0.3)
    
    # Accuracy
    ax2.plot(epochs_ran, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
    ax2.plot(epochs_ran, [a*100 for a in history['val_acc']],   'r-o', label='Validation', markersize=4)
    ax2.set_title('Accuracy per Epoch')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_ylim(0, 105)
    ax2.legend(); ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Training curves saved to results/training_curves.png')

## 5. 📊 Evaluation on Test Set

In [ ]:
# ---- Evaluate best model on test set ----

if not os.path.exists(DATA_DIR):
    print('⚠️  Dataset not found. Cannot evaluate.')
elif not os.path.exists(MODEL_PATH):
    print(f'⚠️  Model not found at {MODEL_PATH}. Run training first.')
else:
    from evaluate import run_evaluation, print_metrics, save_confusion_matrix, save_sample_predictions
    
    # Load best model
    model = BrainTumorCNN(num_classes=2).to(device)
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded model from epoch {checkpoint["epoch"]} '
          f'(val_acc={checkpoint["val_acc"]*100:.2f}%)')
    
    # Run on test set
    preds, true_labels, probs = run_evaluation(model, test_loader, device)
    
    # Print metrics
    metrics = print_metrics(preds, true_labels, class_names)

In [ ]:
# ---- Confusion matrix (inline) ----

if os.path.exists(DATA_DIR) and os.path.exists(MODEL_PATH):
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(true_labels, preds)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted: no', 'Predicted: yes'],
                yticklabels=['Actual: no', 'Actual: yes'],
                linewidths=0.5, linecolor='gray', ax=ax)
    ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold', pad=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Confusion matrix saved to results/confusion_matrix.png')

In [ ]:
# ---- Sample predictions grid ----

if os.path.exists(DATA_DIR) and os.path.exists(MODEL_PATH):
    import torch.nn as nn
    
    model.eval()
    softmax = nn.Softmax(dim=1)
    
    images_shown, preds_shown, labels_shown, confs_shown = [], [], [], []
    
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs_cpu = imgs
            out = softmax(model(imgs.to(device))).cpu()
            ps  = torch.argmax(out, dim=1)
            for i in range(len(lbls)):
                if len(images_shown) >= 12: break
                images_shown.append(imgs_cpu[i])
                preds_shown.append(ps[i].item())
                labels_shown.append(lbls[i].item())
                confs_shown.append(out[i][ps[i].item()].item())
            if len(images_shown) >= 12: break
    
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    fig.suptitle('Test Set Predictions (Green=Correct, Red=Wrong)', fontsize=12, fontweight='bold')
    axes = axes.flatten()
    
    for i, (img, pred, lbl, conf) in enumerate(zip(images_shown, preds_shown, labels_shown, confs_shown)):
        img_disp = denormalize(img).permute(1,2,0).numpy()
        axes[i].imshow(img_disp)
        axes[i].axis('off')
        color = 'green' if pred == lbl else 'red'
        axes[i].set_title(f'Pred: {class_names[pred]} ({conf*100:.1f}%)\nTrue: {class_names[lbl]}',
                          color=color, fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 6. 🔍 Single Image Prediction

In [ ]:
# ---- Test prediction on a single image ----
# Change IMAGE_PATH to any MRI image you want to test

IMAGE_PATH = str(Path(DATA_DIR) / 'yes' / 'Y10.jpg')  # Example — change as needed

if not os.path.exists(DATA_DIR):
    print('⚠️  Dataset not found.')
elif not os.path.exists(MODEL_PATH):
    print('⚠️  Model not found. Train first.')
elif not os.path.exists(IMAGE_PATH):
    print(f'⚠️  Image not found: {IMAGE_PATH}')
    print('   Update IMAGE_PATH to a valid image.')
else:
    from predict import predict_image
    result = predict_image(
        image_path=IMAGE_PATH,
        model_path=MODEL_PATH,
        save_result=True,
        save_dir=RESULTS_DIR,
    )
    
    # Show inline
    pred_img_path = os.path.join(RESULTS_DIR, f'prediction_{Path(IMAGE_PATH).stem}.png')
    if os.path.exists(pred_img_path):
        img = Image.open(pred_img_path)
        plt.figure(figsize=(10, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## 7. 📚 Course Knowledge Point Summary

This cell summarises how each course chapter connects to our implementation.

In [ ]:
summary = [
    ('Chapter 2', 'Machine Learning Overview',
     'CrossEntropyLoss, mini-batch SGD, train/val split, overfitting detection'),
    ('Chapter 3', 'Linear Models',
     'Softmax output in classifier head, Cross-Entropy loss derivation'),
    ('Chapter 4', 'Feedforward Neural Networks',
     'ReLU activations in each Conv block, fully-connected layers'),
    ('Chapter 5', 'Convolutional Neural Networks',
     'Conv2d layers, feature maps, 3×3 filters, MaxPool2d downsampling'),
    ('Chapter 7', 'Optimisation & Regularisation',
     'Adam optimizer, BatchNorm2d, Dropout(0.5), Data Augmentation, Early Stopping, StepLR'),
]

print('=' * 70)
print('  COURSE KNOWLEDGE POINT CONNECTION')
print('=' * 70)
for chapter, topic, implementation in summary:
    print(f'  {chapter}: {topic}')
    print(f'    → {implementation}')
    print()
print('=' * 70)

---

## 8. Conclusion

### What we achieved:
- Built a **custom CNN from scratch** using PyTorch for brain tumor detection
- Applied **data augmentation** to combat overfitting on a small dataset
- Used **Adam optimizer** with learning rate scheduling for efficient training
- Implemented **BatchNorm** and **Dropout** for stable and generalisable training
- Achieved **>85% accuracy** on the test set

### Possible Improvements:
- Transfer learning (ResNet, EfficientNet) for higher accuracy
- Grad-CAM visualisation to highlight which brain regions trigger detection
- Larger dataset (Kaggle Brain Tumor MRI Dataset with 4 classes)
- Multi-class classification (glioma, meningioma, pituitary, no tumor)

### References:
1. Navoneel Chakrabarty, *Brain MRI Images for Brain Tumor Detection*, Kaggle, 2019
2. Ngo & Krebs, *Neural Networks and Deep Learning* (course textbook)
3. PyTorch Documentation — https://pytorch.org/docs/stable/index.html
4. LeCun et al., *Gradient-Based Learning Applied to Document Recognition*, IEEE 1998
